In [1]:
import os
import gzip
import pickle
from datasets import load_dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tabix
from sklearn.preprocessing import MinMaxScaler

In [2]:
approach = 'attention'

# Extract layer information

``` python
results:dict x12 keys (heads)
results[0]:list x2000 tuples
results[0][0]:tuple x2 lists
results[0][0][0]:list x510 'scores'
results[0][0][1]:list x508 '6mers'
results:dict = {head0: [(['score']x510, ['6mer']x508)]x2000}
```

In [ ]:
def kmer2seq(kmers_list:list) -> str:
    first:bool = True
    for token in kmers_list:
        if first:
            sequence:str = token
            first = False
        elif (token != '[SEP]' and token != '[PAD]'):
            sequence += token[-1]
        else:
            break
    return(sequence)

# Create a dictionary with all the data
results_dict:dict = {}
for layer in range(12):
    with open(f'/enhancer_scores_random_init/examples_scores_{approach}_1000_layer{layer}.p', 'rb') as f:
        results:dict = pickle.load(f)
    for head in range(12):
        for example in range(len(results[head])):
            sequence:str = kmer2seq(results[head][example][1])
            if layer == 0 and head ==0:
                results_dict[sequence] = {}
                results_dict[sequence]['kmers'] = [kmer for kmer in results[head][example][1] if (kmer != '[SEP]' and kmer != '[PAD]')]
            kmers_vector_length = len(results_dict[sequence]['kmers'])
            results_dict[sequence][f'layer{layer}-head{head}'] = np.array(results[head][example][0][1:kmers_vector_length+1])

# Save the dictionary
with open(f'/enhancer_results_random_init/enhancer_scores_dictionary_{approach}_random_init.pkl', 'wb') as f:
    pickle.dump(results_dict, f)
del(results_dict, results, layer, head, example, sequence, kmers_vector_length)

``` python
results_dict:dict x2000 keys (sequences key)
results_dict['ACGT']:dict x145 keys (x1 kmers + x144 layers[0-11]-heads[0-11])
results_dict['ACGT']['kmers']:list x50-510 (sequence-dependent kmers)
results_dict['ACGT']['layer0-head0']:list x50-510(sequence-dependent kmers scores)
results_dict:dict = {'ACGT': {'kmers': [kmers]x50-510, 'layer0-head0': [score]x50-510}}
```

# Update results dictionary

In [ ]:
# Open previous files
## Enhancer scores
with open(f'/enhancer_results_random_init/enhancer_scores_dictionary_{approach}_random_init.pkl', 'rb') as f:
    results_dict:dict = pickle.load(f)
## Blat results
blat_query_seq_results_filtered = pd.read_csv(f'/enhancer_results/blat/blat_query_seq_results_filtered_enhancer.csv')

# Update results
sequences:list = list(results_dict.keys())
blat_sequences:list = list(blat_query_seq_results_filtered['Sequence'])
for sequence in sequences:
    ## Remove sequences that couldn't be aligned with Blat
    if sequence not in blat_sequences:
        results_dict.pop(sequence)
        continue

# Save the updated results dictionary
with open(f'/enhancer_results_random_init/enhancer_scores_dictionary_{approach}_random_init_updated.pkl', 'wb') as f:
    pickle.dump(results_dict, f)

del(sequences, sequence, blat_sequences, f, results_dict, blat_query_seq_results_filtered)

# Analysis

``` python
results_dict:dict x1994 keys (sequences key)
results_dict['ACGT']:dict x145 keys (x1 kmers + x144 layers[0-11]-heads[0-11])
results_dict['ACGT']['kmers']:list x50-510 (sequence-dependent kmers)
results_dict['ACGT']['layer0-head0']:list x50-510 (sequence-dependent kmers scores)
results_dict:dict = {'ACGT': {'kmers': [kmers]x50-510, 'layer0-head0': [score]x50-510}}
```

In [ ]:
def transform_values(values):
    new_values:list = np.array([np.mean(values[i:i+6]) for i in range(len(values)-5)])
    return(new_values)

# Open previous results
## Enhancer scores
with open(f'/enhancer_results_random_init/enhancer_scores_dictionary_{approach}_random_init_updated.pkl', 'rb') as f:
    results_dict:dict = pickle.load(f)
## Blat results
blat_query_seq_results_filtered:pd.DataFrame = pd.read_csv(f'/enhancer_results/blat/blat_query_seq_results_filtered_enhancer.csv')

# Get the list of sequences and their indexes in the blat dataframe
sequences:list = list(results_dict.keys())
indexes:list = [blat_query_seq_results_filtered[blat_query_seq_results_filtered['Sequence'] == sequence].index[0] for sequence in sequences]

del(f)

In [ ]:
# Extract conservation information (PhyloP)
phyloP:tabix = tabix.open('/databases/conservation/hg38.phyloP100way.sorted.combined.bed.gz')
for idx, sequence in zip(indexes, sequences):
    chrom:str = blat_query_seq_results_filtered['Chrom'][idx]
    start:int = int(blat_query_seq_results_filtered['Start'][idx])
    end:int = int(blat_query_seq_results_filtered['End'][idx])
    ## Retrieve information from phyloP database with tabix
    records:object = phyloP.querys(f"chr{chrom}:{start}-{end}")
    values:list = []
    for record in records:
        for position in range(int(record[1]), int(record[2])):
            if position >= start and position < end:
                values.append(float(record[3]))
    kmers_vector_length:int = len(results_dict[sequence]['kmers'])
    ## Add this layer to the original one
    if len(values) != kmers_vector_length+5: #Very repetitive regions may not have phyloP or an incomplete track
        results_dict[sequence]['phyloP'] = np.array(np.full(kmers_vector_length, np.nan))
    else:
        results_dict[sequence]['phyloP'] = transform_values(values)

del(phyloP, idx, sequence, chrom, start, end, records, values, record, position, kmers_vector_length)


In [ ]:
# Annotate TSS regions
tss_db:tabix = tabix.open("/databases/refTSS4.1/refTSS_v4.1_human_coordinate.hg38.bed.txt.gz")
for idx, sequence in zip(indexes, sequences):
    chrom:str = blat_query_seq_results_filtered['Chrom'][idx]
    start:int = int(blat_query_seq_results_filtered['Start'][idx])
    end:int = int(blat_query_seq_results_filtered['End'][idx])
    ## Retrieve information from refTSS database with tabix
    records:object = tss_db.querys(f"chr{chrom}:{start}-{end}")
    tss_region:list = []
    kmers_vector_length:int = len(results_dict[sequence]['kmers'])
    for record in records:
        tss_region.extend([i for i in range(int(record[1]), int(record[2])+1)]) #Can be more than one TSS described for each region
    if len(tss_region) == 0: #If there isn't a TSS in the region, fill the layer with NAs
        results_dict[sequence]['TSS'] = np.array(np.full(kmers_vector_length, np.nan))
    else:
        query_region = [i for i in range(start, end+1)]
        tss_vector = []
        for pos in query_region:
            tss_vector.extend([1 if pos in tss_region else 0])
        ## Add this layer to the original one
        results_dict[sequence]['TSS'] = transform_values(tss_vector[1:])

del(tss_db, idx, sequence, chrom, start, end, records, tss_region, record, pos, query_region, tss_vector)

In [8]:
# Annotate GC content
for sequence in sequences:
    kmer_list:list = results_dict[sequence]['kmers']
    gc_vector:list = []
    for kmer in kmer_list:
        ## Change C and G to 1, and A and T to 0
        gc_sequence = kmer.replace('C', '1').replace('G', '1').replace('A', '0').replace('T', '0')
        gc_sequence = [int(i) for i in gc_sequence]
        gc_vector.append(np.mean(gc_sequence))
    
    ## Add this layer to the original one
    results_dict[sequence]['GC'] = np.array(gc_vector)

del(sequence, kmer_list, gc_vector, kmer, gc_sequence)

In [ ]:
# Extract TF information (JASPAR)
jaspar:tabix = tabix.open('/databases/JASPAR/JASPAR_TFs_hg38.sorted.bed.gz')
for idx, sequence in zip(indexes, sequences):
    chrom:str = blat_query_seq_results_filtered['Chrom'][idx]
    start:int = int(blat_query_seq_results_filtered['Start'][idx])
    end:int = int(blat_query_seq_results_filtered['End'][idx])
    ## Retrieve information from JASPAR database with tabix
    records:object = jaspar.querys(f"chr{chrom}:{start}-{end}")
    tf_positions:dict = {} #Each entry will be the genomic regions for each transcript factor
    for record in records:
        tf:str = record[3]
        if tf in tf_positions.keys(): #Each TF can be detected in different regions, but we want an entry per TF
            tf_positions[tf].extend([i for i in range(int(record[1]), int(record[2])+1)])
        else:
            tf_positions[tf] = [i for i in range(int(record[1]), int(record[2])+1)]
    ## Add each TF as a new key in the sequences dictionary
    region:list = [i for i in range(start, end+1)]
    for tf in tf_positions:
        region_scores:list = []
        for pos in region:
            region_scores.extend([1 if pos in tf_positions[tf] else 0])
        results_dict[sequence][tf] = transform_values(region_scores[1:])

del(jaspar, idx, sequence, chrom, start, end, records, tf, record, tf_positions, region, region_scores, pos)

In [ ]:
# Since not all the TF are detected in all the sequences, we need to fill those with NAs
TF_families:pd.DataFrame = pd.read_csv('/databases/JASPAR/JASPAR_TF_families.csv', sep=",").drop_duplicates()
for sequence in results_dict:
    for tf in TF_families['TF']:
        if tf not in results_dict[sequence].keys():
            kmers_vector_length:int = len(results_dict[sequence]['kmers'])
            results_dict[sequence][tf] = np.array(np.full(kmers_vector_length, np.nan))

del(sequence, tf)

In [11]:
# Combine TF matrix by families
uniqueTF_families:list = list(TF_families['Family'].unique())
for sequence in results_dict:
    for family in uniqueTF_families:
        idx:pd.Index = TF_families[TF_families['Family']==family].index
        results_dict[sequence][family] = np.nansum(np.vstack([results_dict[sequence][TF_families['TF'][i]] for i in idx]), axis=0)
        
del(sequence, family, idx, uniqueTF_families)

In [12]:
# Remove single TFs from the results dictionary
for sequence in results_dict:
    for tf in TF_families['TF']:
        try:
            results_dict[sequence].pop(tf)
        except KeyError:
            continue

del(sequence, tf, TF_families)

In [ ]:
# Check if the sequence belong to a repeat element
repeat_db:tabix = tabix.open("/databases/repeatmasker/repeat_masker_hg38.bed.gz")
repeat_families:pd.DataFrame = pd.read_csv('/databases/repeatmasker/repeat_families.csv', sep=",")
for idx, sequence in zip(indexes, sequences):
    chrom:str = blat_query_seq_results_filtered['Chrom'][idx]
    start:int = int(blat_query_seq_results_filtered['Start'][idx])
    end:int = int(blat_query_seq_results_filtered['End'][idx])
    ## Create a list for each repeat element
    repeat_positions:dict = {} #Each entry will be the genomic regions for repeat element
    for repeat in repeat_families['Family']:
        repeat_positions[repeat] = []
    ## Retrieve information from repeat masker database with tabix
    records:object = repeat_db.querys(f"{chrom}:{start}-{end}")
    for record in records:
        try: #There are some repeats like LTR? or DNA? that we are not considering
            repeat_positions[record[4]].extend([i for i in range(int(record[1]), int(record[2])+1)])
        except KeyError:
            continue
    ## Add each repeat element as a new key in the sequences dictionary
    region:list = [i for i in range(start, end+1)]
    for repeat in repeat_positions:
        region_scores:list = []
        for pos in region:
            region_scores.extend([1 if pos in repeat_positions[repeat] else 0])
        results_dict[sequence][repeat] = transform_values(region_scores[1:])

del(repeat_db, idx, sequence, chrom, start, end, records, repeat, record, repeat_positions, region, region_scores, pos)

In [14]:
# Combine repeats by families
unique_repeat_families:list = list(repeat_families['Superfamily'].unique())
for sequence in results_dict:
    for family in unique_repeat_families:
        idx:pd.Index = repeat_families[repeat_families['Superfamily']==family].index
        results_dict[sequence][family] = np.nansum(np.vstack([results_dict[sequence][repeat_families['Family'][i]] for i in idx]), axis=0)
        
del(sequence, family, idx, unique_repeat_families)

In [15]:
# Remove single repeats from the results dictionary
for sequence in results_dict:
    for repeat in repeat_families['Family']:
        try:
            results_dict[sequence].pop(repeat)
        except KeyError:
            continue

del(sequence, repeat, repeat_families)

In [16]:
# Add positional features
for sequence in results_dict:
    kmers_vector_length:int = len(results_dict[sequence]['kmers'])
    results_dict[sequence]['position'] = np.array([i for i in range(kmers_vector_length)])

del(sequence)

In [17]:
# Extract feature list to check
features_list:list = list(results_dict[list(results_dict.keys())[0]].keys())
features_list = [i for i in features_list if (i not in ['sequence', 'kmers', 'phyloP', 'TSS', 'GC', 'position']) and (not i.startswith('layer'))]

# Initialize the feature dict
features_dict:dict = {}
for feature in results_dict[list(results_dict.keys())[0]]:
    if feature not in []:
        features_dict[feature] = []

# Filter features present in < 5% of sequences
remove_features:list = []
for key in results_dict:
    for feature in features_list:
        features_dict[feature].append(0 if max(results_dict[key][feature]) == 0 else 1)
for feature in features_list:
    if (sum(features_dict[feature]) / len(features_dict[feature])) < 0.05:
        remove_features.append(feature)

# Remove lowly expressed features
for key in results_dict:
    for feature2remove in remove_features:
        del results_dict[key][feature2remove]

del(features_list, features_dict, feature, remove_features, key, feature2remove)

In [ ]:
with open(f'/enhancer_results_random_init/enhancer_scores_dictionary_{approach}_random_init_updated_annotations.pkl', 'wb') as output:
    pickle.dump(results_dict, output, protocol=pickle.HIGHEST_PROTOCOL)

del(output)

# Prepare results for correlation analysis

In [ ]:
# Open previous results
## Enhancer scores
with open(f'/enhancer_results_random_init/enhancer_scores_dictionary_{approach}_random_init_updated_annotations.pkl', 'rb') as f:
    results_dict:dict = pickle.load(f)

# Convert dictionary into list of dictionaries
df:list = []
for key, subdict in results_dict.items():
    dict_by_row:dict = {'sequence': key}
    for subkey, values_list in subdict.items():
        dict_by_row[subkey] = ','.join(map(str, values_list))
    df.append(dict_by_row)

# Create DataFrame
df:pd.DataFrame = pd.DataFrame(df)

# Save the dataframe as a csv file with semicolon separator
with gzip.open(f'/enhancer_results_random_init/enhancer_scores_dictionary_{approach}_random_init_updated_annotations_corformat_nominmax.csv.gz', 'wt', newline='', encoding='utf-8') as f:
    df.to_csv(f, sep=';', index=False)

del(key, subdict, subkey, dict_by_row, values_list, df, f)